# **Fine-Tuning BERT for POS Tagging and Chunking using Token Classification**

**Install & Import Libraries**

In [3]:
dataset_name = "conll2003"

In [ ]:
!pip install datasets==2.19.0

In [ ]:
!pip install seqeval -q

**Load Dataset**

In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003")

print(dataset)

**Label Names**

In [7]:
label_names = dataset["train"].features["ner_tags"].feature.names
print(label_names)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


**Tokenizer**

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

**Tokenization + Label Alignment**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # Ignore special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)  # Ignore subword tokens

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

**Model Setup**

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_names)
)

**Training Setup**

In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=0.1,
    weight_decay=0.01
)

**Evaluation Metrics**

In [13]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",   # ✅ ADD THIS
        max_length=128,         # ✅ ADD THIS
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

In [15]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)


In [21]:
from seqeval.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(p):
    predictions, labels = p

    predictions = predictions.argmax(axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        curr_pred = []
        curr_lab = []

        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                curr_pred.append(label_names[p_val])   # ✅ FIXED
                curr_lab.append(label_names[l_val])    # ✅ FIXED

        true_predictions.append(curr_pred)
        true_labels.append(curr_lab)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "accuracy": accuracy_score(true_labels, true_predictions),
    }

**Trainer**

In [23]:
from transformers import Trainer, TrainingArguments

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

**Train Model**

In [ ]:
trainer.train()

**Evaluate Model**

In [24]:
trainer.evaluate()

{'eval_loss': 0.33155152201652527,
 'eval_model_preparation_time': 0.0098,
 'eval_precision': 0.5368178829717292,
 'eval_recall': 0.5502021563342318,
 'eval_f1': 0.5434276206322795,
 'eval_accuracy': 0.9242055218907702,
 'eval_runtime': 25.9852,
 'eval_samples_per_second': 125.071,
 'eval_steps_per_second': 7.851}

**Inference**

In [29]:
import torch

def predict(sentence):
    tokens = sentence.split()

    # 🔹 Step 1: Tokenize WITHOUT tensors
    tokenized_inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True
    )

    word_ids = tokenized_inputs.word_ids()

    # 🔹 Step 2: Convert to tensors
    inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # 🔹 Step 3: Prediction
    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)

    predicted_labels = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        elif word_idx != previous_word_idx:
            predicted_labels.append(label_names[predictions[0][idx].item()])
        previous_word_idx = word_idx

    return list(zip(tokens, predicted_labels))

In [30]:
sentence = "John works at Google in California"
print(predict(sentence))

[('John', 'O'), ('works', 'O'), ('at', 'O'), ('Google', 'B-LOC'), ('in', 'O'), ('California', 'B-LOC')]


**Comparison**

**🔹 POS Tagging vs Chunking**

**POS Tagging**

- Works at word level
- Identifies grammar (noun, verb, etc.)
- Easier

**Chunking**

- Works at phrase level
- Groups words (noun phrase, verb phrase)
- More complex

**Observations**

- Tokenization splits words into subwords
- Label alignment is important for correct training
- BERT improves accuracy due to context understanding
- Chunking is harder than POS tagging
- Ignoring special tokens (-100) is important

**FINAL PIPELINE**

Raw Text → Tokenization → Label Alignment → Model Training → Evaluation → Inference